In [18]:
import json
import warnings
from pathlib import Path
from pandas.errors import PerformanceWarning

from nhs_waiting_lists import (
    __app_name__,
)
from nhs_waiting_lists.utils.xdg import XDGBasedir

warnings.simplefilter(action="ignore", category=PerformanceWarning)
import pandas as pd
from nhs_waiting_lists.core.waiting_list_db import WaitingListsDB
from scripts.excel_parsing_olde.parse_nhs_data import load_data_to_database2
from nhs_waiting_lists.utils.loader_utils import clean_wtt_rows

from nhs_waiting_lists.utils.loader_utils import fiscal_to_calendar, parse_rtt_period

db = WaitingListsDB()


In [19]:

project_root = Path(XDGBasedir.get_data_dir(__app_name__))
with open(project_root / "files/downloadsrtt-waiting-times.jsonl") as f:
    is_looping = True
    for line in f:
        data = json.loads(line)
        for foo in data["files"]:
            print(f"file is {project_root / 'files' / foo['path']}")

            period = foo["period"]

            if period < "2021-04-01":
                continue

            if period < "2017-10":
                """
                olde format with pre-column header front matter
                """
                df = pd.read_csv(
                    project_root / 'files' / foo['path'],
                    skiprows=2
                )
                df["Period"] = df.apply(fiscal_to_calendar, axis=1)
            else:
                df = pd.read_csv(
                    project_root / 'files' / foo['path'],
                    skiprows=0,
                    converters={"Period": parse_rtt_period}
                )

            df = clean_wtt_rows(df)

            bad_rows = df[
                (df["diff_total"].abs() > 0.01) | (df["diff_total_all"].abs() > 0.01)
                ]

            if len(bad_rows) > 0:
                # print(f"Bad rows: {bad_rows}")
                is_looping = False
                break  # break out of the inner loop

            cols_to_drop = ["wait_sum", "diff_total", "diff_total_all"]
            df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

            # print(f"columns {df.columns=}")

            load_data_to_database2(df, "all_rtt", db.get_connection())
            # df.to_sql(
            #     name="all_rtt",
            #     con=conn,
            #     if_exists="replace",
            #     index=False
            # )
        if not is_looping:
            break  # break out of outer loop

bad_rows[:5]


file is /home/tomhodder/.local/state/nhs_waiting_lists/files/rtt-waiting-times/Full-CSV-data-file-Aug25-ZIP-4M-71939.zip
Loaded 18531 rows into all_rtt
file is /home/tomhodder/.local/state/nhs_waiting_lists/files/rtt-waiting-times/Full-CSV-data-file-Aug25-ZIP-4M-71939.zip
Loaded 18531 rows into all_rtt
file is /home/tomhodder/.local/state/nhs_waiting_lists/files/rtt-waiting-times/Full-CSV-data-file-Jul25-ZIP-4M-92707.zip
Loaded 18603 rows into all_rtt
file is /home/tomhodder/.local/state/nhs_waiting_lists/files/rtt-waiting-times/Full-CSV-data-file-Jun25-ZIP-4M-94367.zip
Loaded 18542 rows into all_rtt
file is /home/tomhodder/.local/state/nhs_waiting_lists/files/rtt-waiting-times/Full-CSV-data-file-May25-ZIP-4M-32711.zip
Loaded 18460 rows into all_rtt
file is /home/tomhodder/.local/state/nhs_waiting_lists/files/rtt-waiting-times/Full-CSV-data-file-Apr25-ZIP-4M-77252.zip
Loaded 18531 rows into all_rtt
file is /home/tomhodder/.local/state/nhs_waiting_lists/files/rtt-waiting-times/Full-CSV-

,period,provider_org_code,rtt_part_type,treatment_function_code,gt_00_to_01_weeks,gt_01_to_02_weeks,gt_02_to_03_weeks,gt_03_to_04_weeks,gt_04_to_05_weeks,gt_05_to_06_weeks,...,gt_101_to_102_weeks,gt_102_to_103_weeks,gt_103_to_104_weeks,gt_104_weeks,patients_with_unknown_clock_start_date,total,total_all,wait_sum,diff_total,diff_total_all


In [20]:

df.query("provider_org_code == 'A4M8P' and rtt_part_type == 'Part_2'")


,period,provider_org_code,rtt_part_type,treatment_function_code,gt_00_to_01_weeks,gt_01_to_02_weeks,gt_02_to_03_weeks,gt_03_to_04_weeks,gt_04_to_05_weeks,gt_05_to_06_weeks,...,gt_98_to_99_weeks,gt_99_to_100_weeks,gt_100_to_101_weeks,gt_101_to_102_weeks,gt_102_to_103_weeks,gt_103_to_104_weeks,gt_104_weeks,patients_with_unknown_clock_start_date,total,total_all
24,2024-04,A4M8P,Part_2,C_100,46.0,106.0,62.0,69.0,47.0,83.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1349
25,2024-04,A4M8P,Part_2,C_101,13.0,35.0,25.0,16.0,14.0,22.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,799
26,2024-04,A4M8P,Part_2,C_110,13.0,13.0,16.0,13.0,7.0,22.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,452
27,2024-04,A4M8P,Part_2,C_120,30.0,23.0,45.0,25.0,12.0,25.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,589
28,2024-04,A4M8P,Part_2,C_160,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3
29,2024-04,A4M8P,Part_2,C_170,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
30,2024-04,A4M8P,Part_2,C_301,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12
31,2024-04,A4M8P,Part_2,C_502,3.0,21.0,28.0,16.0,11.0,12.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,522
32,2024-04,A4M8P,Part_2,C_999,108.0,201.0,178.0,142.0,91.0,171.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3830
33,2024-04,A4M8P,Part_2,X02,3.0,2.0,2.0,2.0,0.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,92
